In [1]:
# ----- 참고글 -----
# https://sanghyu.tistory.com/90
# http://www.gisdeveloper.co.kr/?p=8615
# 미니배치 관련 https://velog.io/@ksj5738/미니배치-학습
# collate_fn 관련 https://www.notion.so/PyTorch-collate_fn-feat-Dataset-vs-DataLoader-b1f5ceb5bac94ba78bf016fda3679a7f

In [41]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random

In [3]:
# ----- Dataset -----
# Dataset은 input feature x와 label y를 input으로 받아 저장한다.
# 아래와 같은 메소드가 있다.
    # __ init__ : 파일을 읽기 위한 디렉토리/파일을 지정하고 파일을 가공하기
    # __ getitem__ : dataset의 특정 인덱스에 해당하는 데이터 얻기
    # __ len__ : dataset 크기

In [4]:
# 임의의 학습 데이터셋 만들기
# x : 크기가 3인 리스트 10개
# y : 0,1로 이루어진 길이가 10인 리스트
class MyDataset(Dataset):

    def __init__(self):
        x = np.random.rand(10, 3) 
        self.x = [x[i].tolist() for i in range(10)]
        y = np.random.randint(low=0, high=2, size=(10,))
        self.y = [y[i] for i in range(10)]

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [5]:
# 데이터셋 정의
dataset = MyDataset()

In [6]:
# 전체 데이터셋 길이
len(dataset)

10

In [7]:
# 첫번째 데이터 확인
dataset[1]

([0.6426445545801128, 0.6267456316966776, 0.7779337793171849], 1)

In [8]:
# ----- DataLoader -----
# DataLoader는 batch 기반으로 모델을 학습시키기 위해 dataset을 input으로 받아 batch size로 슬라이싱하여 모델로 전달하는 역할을 한다.
# 기본 파라미터에는 아래와 같은 것들이 있다.
    # batch_size (default=1) : 배치 크기
    # shuffle : 데이터를 섞어서 사용할건지 여부
    # num_workers : 데이터 로딩에 사용하는 subprocess 개수 (분산처리 시 필요)
    # drop_last : batch_size에 따라 마지막 batch의 길이가 달라질 수 있는데, 이 경우에 마지막 batch를 사용할건지 여부
    # collate_fn : 데이터를 customize하는 작업. dataloader의 batch 단위로 input을 process하기 위한 작업. 주로 zero-padding이나 Variable Size 데이터 등 데이터 사이즈를 맞추기 위해 많이 사용한다.

In [9]:
dataloader1 = DataLoader(dataset, batch_size=5, shuffle=False)

In [10]:
# 데이터 확인 (첫번째 배치)
x, y = next(iter(dataloader1))

In [11]:
x

[tensor([0.8870, 0.6426, 0.7333, 0.0104, 0.4701], dtype=torch.float64),
 tensor([0.1633, 0.6267, 0.8459, 0.9530, 0.3109], dtype=torch.float64),
 tensor([0.2357, 0.7779, 0.0115, 0.0303, 0.0131], dtype=torch.float64)]

In [12]:
y

tensor([1, 1, 1, 1, 1])

In [13]:
# collate_fn 사용해보기
def collate_fn(batch):
    data_list, label_list = [], []
    for _data, _label in batch:
        data_list.append(_data)
        label_list.append(_label)
    return torch.Tensor(data_list), torch.LongTensor(label_list)

In [14]:
dataloader2 = DataLoader(dataset, batch_size=5, shuffle=False, collate_fn=collate_fn)

In [15]:
# collate_fn을 사용한 후 데이터 확인 (첫번째 배치)
x, y = next(iter(dataloader2))

In [16]:
x

tensor([[0.8870, 0.1633, 0.2357],
        [0.6426, 0.6267, 0.7779],
        [0.7333, 0.8459, 0.0115],
        [0.0104, 0.9530, 0.0303],
        [0.4701, 0.3109, 0.0131]])

In [17]:
y

tensor([1, 1, 1, 1, 1])

In [52]:
# 또는 dataset을 구성할 때에 collate_fn을 직접 삽입할 수도 있다.
# 다른 예시
# x: 길이가 5~10 사이인 텐서 20개
# y: 0~1 사이의 값 20개
class MyDataset(Dataset):

    def __init__(self, csv_file=None, root_dir=None):

        #self.csv_file = csv_file
        #self.root_dir = root_dir

        n_dataset = 20
        
        self.x_data = []
        
        for _ in range(n_dataset):
            random_length = random.randint(5,10)
            self.x_data.append(np.random.randint(0,255,random_length))
        
        self.y_data = torch.rand(n_dataset, 1)
    
    
    def __len__(self): 
        return len(self.x_data)

    
    def __getitem__(self, idx): 
        x = torch.FloatTensor(self.x_data[idx])
        y = torch.FloatTensor(self.y_data[idx])
        return x, y
   

    # 서로 길이가 다른 inpout에 zero-padding을 추가해 길이 맞춰주기
    def collate_fn(self, data):
        
        max_len = 10
        batch = []
        for x, y in data:
            x_padded = torch.cat([x, torch.zeros(max_len - x.shape[0])])
            batch.append(x_padded)
        return torch.stack(batch)

    
dataset = MyDataset()

dataloader = DataLoader(dataset, 
                        batch_size=5, 
                        collate_fn=dataset.collate_fn, 
                        shuffle=True)
# dataloader = DataLoader(dataset, 
#                         batch_size=25, 
#                         shuffle=True) # collate_fn 지정 안하는 경우 사이즈가 맞지 않는다는 에러가 발생

In [57]:
# 각 배치에 대해서 데이터 확인
for i in range(len(dataloader)):
    print(i, next(iter(dataloader)))

0 tensor([[221., 102., 102., 146., 199.,   4.,  91.,   0.,   0.,   0.],
        [105., 113., 241., 174., 241., 114., 148., 196.,  89.,   0.],
        [230.,  54., 122., 224.,  84.,   0.,   0.,   0.,   0.,   0.],
        [ 41., 179., 224., 145.,  26.,  92.,   0.,   0.,   0.,   0.],
        [240., 166.,  63.,  57.,  36., 106., 131., 226., 185.,   0.]])
1 tensor([[138., 179., 125., 170., 176.,  54., 112.,   0.,   0.,   0.],
        [ 40.,  67.,  26., 194., 234.,   0.,   0.,   0.,   0.,   0.],
        [230.,  54., 122., 224.,  84.,   0.,   0.,   0.,   0.,   0.],
        [232., 127., 149.,  70., 176.,  48.,  30.,   0.,   0.,   0.],
        [105., 113., 241., 174., 241., 114., 148., 196.,  89.,   0.]])
2 tensor([[ 23., 211., 188., 159., 117., 103., 212., 141., 135.,   0.],
        [230.,  54., 122., 224.,  84.,   0.,   0.,   0.,   0.,   0.],
        [138., 179., 125., 170., 176.,  54., 112.,   0.,   0.,   0.],
        [232., 127., 149.,  70., 176.,  48.,  30.,   0.,   0.,   0.],
        [226

In [58]:
# 마지막 배치에 대해서만 데이터 확인
for i, samples in enumerate(dataloader):
    if i==len(dataloader)-1:
        print(samples.shape)
        print(samples)

torch.Size([5, 10])
tensor([[221., 102., 102., 146., 199.,   4.,  91.,   0.,   0.,   0.],
        [ 45., 114.,  87., 233., 145., 110.,  62.,   0.,   0.,   0.],
        [150., 161., 119.,  23., 110., 106., 129.,  97., 127.,  51.],
        [168., 179., 211., 222., 154.,   0.,   0.,   0.,   0.,   0.],
        [ 40.,  67.,  26., 194., 234.,   0.,   0.,   0.,   0.,   0.]])


In [59]:
# 각 배치에 대해서 데이터 확인
from tqdm.auto import tqdm
tqdm_bar = tqdm(enumerate(dataloader))
for i, (samples) in tqdm_bar:
    print(i, samples)

0 tensor([[ 74.,  76., 201.,  47., 121.,   0.,   0.,   0.,   0.,   0.],
        [173.,  52., 192.,  41., 219.,  71.,  70.,   1.,  59.,  68.],
        [240., 166.,  63.,  57.,  36., 106., 131., 226., 185.,   0.],
        [221., 102., 102., 146., 199.,   4.,  91.,   0.,   0.,   0.],
        [144.,  69.,   6., 107.,  49.,   0.,   0.,   0.,   0.,   0.]])
1 tensor([[232., 127., 149.,  70., 176.,  48.,  30.,   0.,   0.,   0.],
        [199.,  98., 142., 147.,  61., 172.,   0.,   0.,   0.,   0.],
        [ 40.,  67.,  26., 194., 234.,   0.,   0.,   0.,   0.,   0.],
        [167.,  61., 202., 226.,  82.,  87., 190., 176., 158.,   0.],
        [ 84., 254.,  38.,  99., 143.,   0.,   0.,   0.,   0.,   0.]])
2 tensor([[226.,  50., 126.,  50., 227.,  55.,  47.,   0.,   0.,   0.],
        [ 41., 179., 224., 145.,  26.,  92.,   0.,   0.,   0.,   0.],
        [ 23., 211., 188., 159., 117., 103., 212., 141., 135.,   0.],
        [105., 113., 241., 174., 241., 114., 148., 196.,  89.,   0.],
        [230